In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [216]:
def getPlayerStats(driver, role):
    data = {}
    roles = getRoles(role)
    found = False
    for r in roles:
        try: 
            table = driver.find_element(By.XPATH, f'//*[@id="scout_full_{r}"]/tbody')
            found = True
        except:
            continue
    
    if not found:
        print(roles)
        return {}
    
    type_stat = 'Standard Stats'
    data[type_stat] = {}
    #<tr class="thead over_header thead" data-row="16"> <th aria-label="" data-stat="header_shooting" colspan="3" class=" over_header center">Shooting</th> </tr>
    for row in table.find_elements(By.TAG_NAME, 'tr'):
        th = row.find_element(By.TAG_NAME, 'th')
        #print(f'--{row.get_property("className")} -- {row.text}')
        if row.get_property("className") == "thead over_header thead":
            type_stat = th.text
            data[type_stat] = {}
            #print(type_stat)
            
        #data_desc = th.get_attribute('data-tip')
        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) > 0 and th.text != '':
            value, perc = tds[0].text , tds[1].text
            #f'{th.text} ({data_desc})'
            data[type_stat][th.text] = {'value': value, 'percentile': perc}

    return data

def getPlayerAnag(driver):
    anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    player = anag_div.find_element(By.TAG_NAME, 'h1').text

    anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[2]').text
    anag_elem3 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[3]').text
    anag_elem4 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[4]').text
    anag_elem5 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[5]').text

    anag_elem1_split = anag_elem1.split('▪')
    anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    height, weight = anag_elem2_split[0], anag_elem2_split[1]
    year_birth = anag_elem3.split(' ')[3]
    nat = anag_elem4.split(' ')[2]
    team = anag_elem5.split(' ')[1]
    height,weight,year_birth, nat, team
    return dict(player=player, 
                position=position, 
                #foot=footed, 
                height= height, 
                weight=weight, 
                year_birth=year_birth, 
                nationality=nat, team=team)


def getPlayerRecord(driver, url):
    driver.get(url)
    player = getPlayerAnag(driver)
    stats = getPlayerStats(driver, player['position'])
    player['stats'] = stats
    return player

def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        elif ')' in r:
            rr.append(r[:-1])
        else:
            rr.append(r.strip())


    return rr

def initializeDriver():
    driver = webdriver.Chrome()
    url = 'https://fbref.com/en/'
    driver.get(url)
    cookie_button = driver.find_elements(By.TAG_NAME, 'button')
    for b in cookie_button:
        if b.text == 'Accetta tutto':
            b.click()
    return driver

In [4]:
url = 'https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report'


In [ ]:
def initializeDriver():
    driver = webdriver.Chrome()
    url = 'https://fbref.com/en/'
    driver.get(url)
    cookie_button = driver.find_elements(By.TAG_NAME, 'button')
    for b in cookie_button:
        if b.text == 'Accetta tutto':
            b.click()
    return driver

In [220]:
#cookie_button = driver.find_element(By.XPATH, '//*[@id="bd44e2e6-0aae-43e0-bae8-847a8a5e55a6"]/div[2]/button[2]')
cookie_button = driver.find_elements(By.TAG_NAME, 'button')

#//*[@id="10b57b08-c511-465a-b19d-f1569498078c"]/div[2]/button[2]
#//*[@id="5363d468-fad8-45fc-8284-ea0f1779dec5"]/div[2]/button[2]

for b in cookie_button:
    if b.text == 'Accetta tutto':
        b.click()


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=132.0.6834.111)
Stacktrace:
	GetHandleVerifier [0x00007FF70EF502F5+28725]
	(No symbol) [0x00007FF70EEB2AE0]
	(No symbol) [0x00007FF70ED4510A]
	(No symbol) [0x00007FF70ED1EEA5]
	(No symbol) [0x00007FF70EDC6F87]
	(No symbol) [0x00007FF70EDDFA52]
	(No symbol) [0x00007FF70EDBFD53]
	(No symbol) [0x00007FF70ED8A0E3]
	(No symbol) [0x00007FF70ED8B471]
	GetHandleVerifier [0x00007FF70F27F30D+3366989]
	GetHandleVerifier [0x00007FF70F2912F0+3440688]
	GetHandleVerifier [0x00007FF70F2878FD+3401277]
	GetHandleVerifier [0x00007FF70F01AAAB+858091]
	(No symbol) [0x00007FF70EEBE74F]
	(No symbol) [0x00007FF70EEBA304]
	(No symbol) [0x00007FF70EEBA49D]
	(No symbol) [0x00007FF70EEA8B69]
	BaseThreadInitThunk [0x00007FFBC5DF259D+29]
	RtlUserThreadStart [0x00007FFBC6B2AF38+40]


In [221]:
driver= initializeDriver()
records = []
urls = [
    'https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report',
    'https://fbref.com/en/players/d4c9725f/scout/12229/Theo-Hernandez-Scouting-Report',
    'https://fbref.com/en/players/a0d55a09/scout/12207/Bradley-Barcola-Scouting-Report',
    #'https://fbref.com/en/players/1f44ac21/scout/12192/Erling-Haaland-Scouting-Report',
    'https://fbref.com/en/players/0db169ae/scout/11611/Sandro-Tonali-Scouting-Report',
    'https://fbref.com/en/players/3f5f38fb/scout/12229/Nicolo-Casale-Scouting-Report',
    'https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report',
    'https://fbref.com/en/players/3260690c/scout/12212/Benjamin-Sesko-Scouting-Report'
]
for url in urls:
    print(url)
    records.append(getPlayerRecord(driver,url))


https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report
https://fbref.com/en/players/d4c9725f/scout/12229/Theo-Hernandez-Scouting-Report
https://fbref.com/en/players/a0d55a09/scout/12207/Bradley-Barcola-Scouting-Report
['FW', 'MF']
https://fbref.com/en/players/0db169ae/scout/11611/Sandro-Tonali-Scouting-Report
https://fbref.com/en/players/3f5f38fb/scout/12229/Nicolo-Casale-Scouting-Report
https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report
https://fbref.com/en/players/3260690c/scout/12212/Benjamin-Sesko-Scouting-Report


In [226]:
records[-5]
#scout_full_FW > tbody > tr:nth-child(62)

{'player': 'Bradley Barcola',
 'position': 'FW-MF',
 'height': '188cm,',
 'weight': '63kg',
 'year_birth': '2002',
 'nationality': 'France',
 'team': 'Paris',
 'stats': {}}

In [227]:
writeJson(records,'Dataset/Fbref/prova_records_perc_v2.json')

In [159]:
roles = ['MF (CM-DM)','FW-MF (AM, left)','DF (FB, left)', 'FW-MF', 'DC (DF)']


getRoles(roles[-1])

['DC', 'DF']

In [121]:
p = records[0]


In [ ]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v2.json')
for p in records:
    player_name = p['player']
    prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
            I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
            For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
            Percentile comparison is made between players of same role.
            Your task is to analyze this data and provide a report as follows:

            ### Input Data:
                - Player: {player_name}
                - Position: {p['position']}
                - Year birth: {p['year_birth']}
                - Height: {p['height']}
                - Weight: {p['weight']}
                - Statistics per 90 minutes: 
                {p['stats']}

            ### Output Format:
            Your report should be structured in the following way:
            **Player**: {player_name}
            **Strengths**: 
            Highlight the player's key strengths evident from their playing style.
            **Weaknesses**: 
            Point out areas where the player needs improvement.
            **Summary**:
            A brief summary of the player's overall performance.


            ### Notes for Analysis:
            - Use concise and professional language.
            - The report should be realistic for scouting purposes.
            - Do not generate code or class structures. Focus only on the football analysis.
            - The output must be in plain text, clearly formatted according to the structure above.
            - Do not write the name of the player 
            - Do not include statistics into report
            
            ###Generated Report:"""
    prompts[player_name] = {'prompt': prompt}

writeJson(prompts, 'Descriptions/prova_stats.json')

In [ ]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v2.json')
for p in records:
    player_name = p['player']
    prompts[player_name] = {'prompt':{}}
    for k, stat  in p['stats'].items():
        
        prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
                I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
                For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
                Percentile comparison is made between players who play in same positions.
                Your task is to analyze this data and provide a report as follows:

                ### Input Data:
                    - Player: {player_name}
                    - Position: {p['position']}
                    - Year birth: {p['year_birth']}
                    - Height: {p['height']}
                    - Weight: {p['weight']}
                    - Statistics per 90 minutes about {k}: 
                    {p['stats']}

                ### Output Format:
                Your report should be structured in the following way:
                **Player**: {player_name}
                **Strengths**: 
                Highlight the player's key strengths evident from their playing style.
                **Weaknesses**: 
                Point out areas where the player needs improvement.
                **Summary**:
                A brief summary of the player's overall performance.


                ### Notes for Analysis:
                - Use concise and professional language.
                - The report should be realistic for scouting purposes.
                - Do not generate code or class structures. Focus only on the football analysis.
                - The output must be in plain text, clearly formatted according to the structure above.
                - Do not write the name of the player 
                - Do not include statistics into report
                - Provide report only about data in input and only about {k}
                
                ###Generated Report:"""
        
        prompts[player_name]['prompt'][k] = prompt

writeJson(prompts, 'Descriptions/prova_stats_v2.json')